# Python Decorators — Complete Advanced Guide (with AI/ML Engineering Applications)

This notebook takes you from the fundamentals of decorators all the way to advanced, production-grade patterns used in real-world AI/ML codebases — model registries, caching inference results, retrying flaky API calls, async LLM calls, timing training loops, and more.

**Table of Contents**
1. Functions as First-Class Objects
2. Closures
3. Your First Decorator
4. Preserving Metadata with `functools.wraps`
5. Decorators That Take Arguments
6. Universal Decorator (works with *or* without arguments)
7. Class-Based Decorators
8. Stacking Decorators — Order Matters
9. Built-in Method Decorators (`@staticmethod`, `@classmethod`, `@property`)
10. Caching with `functools.lru_cache`
11. Timing & Profiling Decorator
12. Retry Decorator with Exponential Backoff
13. Logging Decorator
14. Input Validation Decorator
15. Registry Pattern (Model-Zoo style)
16. Singleton Decorator
17. Thread-Safe Decorator
18. Type-Based Dispatch with `functools.singledispatch`
19. Async Decorators
20. Deprecation Warning Decorator
21. Class Decorators
22. Typing Decorators Correctly (`ParamSpec`)
23. Mini Project — Production Inference Pipeline Decorator Stack
24. Pitfalls & Best Practices
25. Cheat Sheet

We build intuition from first principles (functions as objects, closures) up to the exact decorator patterns you'll meet inside real ML codebases (Hugging Face Transformers, PyTorch, FastAPI-based ML services, etc).

## 1. Functions Are First-Class Objects

In Python, functions are objects like any other value. This single fact is *the* foundation that makes decorators possible:

- They can be assigned to variables
- They can be stored in data structures (lists, dicts)
- They can be passed as arguments to other functions
- They can be returned from other functions

Decorators exploit all four of these properties at once.

In [1]:
def welcome():
    return "Welcome to the advanced python course"

# 1. Assign a function to a variable
wel = welcome
print(wel())

# 2. Store functions in a data structure
funcs = {"greet": welcome}
print(funcs["greet"]())

# 3. Pass a function as an argument to another function
def call_it(func):
    return func()

print(call_it(welcome))

Welcome to the advanced python course
Welcome to the advanced python course
Welcome to the advanced python course


## 2. Closures

A **closure** is a function that "remembers" variables from the scope it was created in, even after that outer scope has already finished executing. Decorators are closures: the inner `wrapper` function remembers the original `func` passed into the outer decorator function.

In [2]:
def main_welcome(msg):
    def sub_welcome_method():
        print("Welcome to the advanced python course")
        print(msg)            # <-- 'msg' is captured from the enclosing scope
        print("Please learn these concepts properly")
    return sub_welcome_method     # return the FUNCTION, not its result

greet = main_welcome("Welcome broooo")
greet()              # msg is still remembered, even though main_welcome() already returned

print(greet.__closure__)
print(greet.__closure__[0].cell_contents)

Welcome to the advanced python course
Welcome broooo
Please learn these concepts properly
(<cell at 0x0000016842E48EB0: str object at 0x0000016842E58B30>,)
Welcome broooo


## 3. Your First Decorator

A decorator is simply a function that:
1. Takes a function as input
2. Defines a new function (`wrapper`) that adds some behaviour around it
3. Returns that `wrapper`

`@decorator` written above a function definition is just syntactic sugar for `func = decorator(func)`.

In [3]:
def my_decorator(func):
    def wrapper():
        print("Something is happening BEFORE the function is called")
        func()
        print("Something is happening AFTER the function is called")
    return wrapper

def say_hello():
    print("Hello!")

# Manual decoration (no @ syntax) -- this is literally what @ does for you
say_hello = my_decorator(say_hello)
say_hello()

Something is happening BEFORE the function is called
Hello!
Something is happening AFTER the function is called


In [5]:
# Equivalent using @ syntax sugar
@my_decorator
def say_hello():
    print("Hello!")

say_hello()

Something is happening BEFORE the function is called
Hello!
Something is happening AFTER the function is called


## 4. Preserving Metadata with `functools.wraps`

**Problem:** once wrapped, the function loses its original identity — `__name__`, `__doc__` and its signature all point to `wrapper`, not to the original function. This silently breaks introspection tools that ML frameworks rely on heavily (FastAPI route inspection, Sphinx autodoc, `help()`, `inspect.signature`, pytest fixture resolution, etc).

In [4]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def predict(x):
    """Run model inference on x."""
    return x * 2

print(predict.__name__)   # 'wrapper'  -- WRONG, identity lost
print(predict.__doc__)    # None       -- WRONG, docstring lost

wrapper
None


In [5]:
import functools

def my_decorator(func):
    @functools.wraps(func)        # <-- copies __name__, __doc__, __module__, etc. onto wrapper
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator  # This is equivalent to: predict = my_decorator(predict)
def predict(x):
    """Run model inference on x."""
    return x * 2

print(predict.__name__)        # 'predict'  -- correct
print(predict.__doc__)         # 'Run model inference on x.'
print(predict.__wrapped__)     # direct access to the original, undecorated function

predict
Run model inference on x.
<function predict at 0x0000016842E3CB80>


> **Rule of thumb:** Always use `@functools.wraps(func)` inside every decorator you write — no exceptions.

## 5. Decorators That Take Arguments

To give a decorator its own configuration arguments (e.g. `@repeat(3)`), you need **three** nested levels of functions:

1. **Outer function** — receives the decorator's own arguments
2. **Middle function** (`decorator`) — receives the function being decorated
3. **Inner function** (`wrapper`) — receives the call-time `*args, **kwargs`

In [6]:
import functools

def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(times=3)
def say_hello(name):
    print(f"Hello, {name}!")

say_hello("World")

Hello, World!
Hello, World!
Hello, World!


## 6. Universal Decorator — Works With *and* Without Arguments

Sometimes you want the exact same decorator usable both as `@my_decorator` and as `@my_decorator(arg=value)`. This pattern appears throughout real frameworks (`pytest.fixture`, `click.command`, etc).

In [8]:
import functools

def smart_decorator(func=None, *, prefix="LOG"):
    """Works as @smart_decorator  OR  @smart_decorator(prefix='...')"""
    if func is None:
        # Called as @smart_decorator(prefix=...) -> return a decorator waiting for func
        return functools.partial(smart_decorator, prefix=prefix)

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[{prefix}] calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@smart_decorator
def f1():
    print("f1 running")

@smart_decorator(prefix="DEBUG")
def f2():
    print("f2 running")

f1()
f2()

[LOG] calling f1
f1 running
[DEBUG] calling f2
f2 running


## 7. Class-Based Decorators

A decorator doesn't have to be a function — any callable object works. Implementing a decorator as a class is especially useful when it needs to **maintain state** across calls (e.g. counting how many times a function was called — handy for tracking inference call counts in production).

In [9]:
import functools

class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)   # the class-based version of functools.wraps
        self.func = func
        self.num_calls = 0

    def __call__(self, *args, **kwargs):
        self.num_calls += 1
        print(f"Call #{self.num_calls} to {self.func.__name__}")
        return self.func(*args, **kwargs)

@CountCalls
def run_inference(x):
    return x ** 2

run_inference(2)
run_inference(3)
run_inference(4)
print(f"Total inference calls: {run_inference.num_calls}")

Call #1 to run_inference
Call #2 to run_inference
Call #3 to run_inference
Total inference calls: 3


## 8. Stacking Multiple Decorators — Order Matters

Decorators apply **bottom-up** but execute **top-down**, like wrapping layers of an onion: the decorator closest to the function wraps it first, then the next one wraps *that* wrapper, and so on.

```python
@a
@b
@c
def f(): ...
```

is exactly equivalent to `f = a(b(c(f)))`.

In [10]:
import functools

def bold(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper

def italic(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper

@bold
@italic
def greet(name):
    return f"Hello, {name}"

print(greet("World"))
# italic runs first (closest to the function), then bold wraps that result
# -> <b><i>Hello, World</i></b>

<b><i>Hello, World</i></b>


## 9. Built-in Method Decorators

Python ships with three decorators built specifically for class methods. You'll use these constantly when building ML model / config classes:

- `@staticmethod` — no implicit `self`/`cls`; a regular function namespaced inside the class
- `@classmethod` — receives the class (`cls`) instead of the instance; commonly used for alternate constructors, e.g. `Model.from_pretrained(...)`
- `@property` — turns a method into a (optionally read-only) attribute

In [11]:
class Model:
    registry = {}

    def __init__(self, name, weights=None):
        self.name = name
        self._weights = weights

    @property
    def weights(self):
        """Read-only access to weights, with lazy initialization."""
        if self._weights is None:
            print("Lazily initializing weights...")
            self._weights = [0.0] * 10
        return self._weights

    @staticmethod
    def preprocess(x):
        """Doesn't need access to the instance or class at all."""
        return [v / 255.0 for v in x]

    @classmethod
    def from_config(cls, config: dict):
        """Alternate constructor -- common pattern in HF Transformers, PyTorch Lightning, etc."""
        return cls(name=config["name"])

m = Model.from_config({"name": "resnet18"})
print(m.name)
print(m.weights)                 # triggers lazy init
print(Model.preprocess([0, 128, 255]))

resnet18
Lazily initializing weights...
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.5019607843137255, 1.0]


## 10. Caching with `functools.lru_cache`

In ML pipelines you often re-compute the same expensive thing: feature extraction for a repeated input, tokenizing the same string, or querying an LLM with an identical prompt. `lru_cache` memoizes return values automatically, keyed on the function's arguments.

In [12]:
import functools
import time

@functools.lru_cache(maxsize=128)
def expensive_feature_extraction(text: str):
    """Pretend this is a slow embedding / feature computation."""
    time.sleep(1)                     # simulate expensive work
    return {"length": len(text), "upper": text.upper()}

start = time.perf_counter()
expensive_feature_extraction("hello world")
print(f"First call:  {time.perf_counter() - start:.2f}s")

start = time.perf_counter()
expensive_feature_extraction("hello world")     # cached -> instant
print(f"Second call: {time.perf_counter() - start:.2f}s")

print(expensive_feature_extraction.cache_info())

First call:  1.00s
Second call: 0.00s
CacheInfo(hits=1, misses=1, maxsize=128, currsize=1)


> `functools.cache` (Python 3.9+) is a convenient shorthand for `lru_cache(maxsize=None)` (an unbounded cache).

> **Caution:** never `lru_cache` a function whose arguments include unhashable types like `numpy.ndarray` or `torch.Tensor` directly — convert to a hashable key first (e.g. `.tobytes()` or a tuple).

## 11. Timing & Profiling Decorator

Essential for ML practitioners — measuring training-step time, inference latency, or data-loading time.

In [13]:
import functools
import time

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[TIMER] {func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def train_one_epoch(num_batches=5):
    for _ in range(num_batches):
        time.sleep(0.1)        # simulate one training batch
    return "epoch done"

train_one_epoch()

[TIMER] train_one_epoch took 0.5006s


'epoch done'

## 12. Retry Decorator with Exponential Backoff

When calling external services — an LLM API, a remote feature store, a flaky GPU server — transient failures happen. A retry decorator with exponential backoff is a standard production pattern for making such calls resilient.

In [14]:
import functools
import random
import time

def retry(times=3, delay=0.5, backoff=2, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            current_delay = delay
            last_exc = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as exc:
                    last_exc = exc
                    print(f"Attempt {attempt}/{times} failed: {exc!r}")
                    if attempt < times:
                        time.sleep(current_delay)
                        current_delay *= backoff
            raise last_exc
        return wrapper
    return decorator

@retry(times=4, delay=0.2, exceptions=(ConnectionError,))
def call_llm_api(prompt):
    """Simulates a flaky API call (e.g. to an LLM provider) that fails ~60% of the time."""
    if random.random() < 0.6:
        raise ConnectionError("Simulated transient network error")
    return f"Response to: {prompt}"

random.seed(1)   # seeded only so this demo prints reproducible output -- remove in real code
print(call_llm_api("Summarize this document"))

Attempt 1/4 failed: ConnectionError('Simulated transient network error')


Response to: Summarize this document


## 13. Logging Decorator

Useful for debugging ML pipelines — automatically log every call to a preprocessing or inference function along with its inputs and outputs.

In [15]:
import functools
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("ml_pipeline")

def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.info(f"Calling {func.__name__}(args={args}, kwargs={kwargs})")
        result = func(*args, **kwargs)
        logger.info(f"{func.__name__} returned {result!r}")
        return result
    return wrapper

@log_calls
def normalize(x, scale=1.0):
    return [v * scale for v in x]

normalize([1, 2, 3], scale=0.5)

INFO: Calling normalize(args=([1, 2, 3],), kwargs={'scale': 0.5})


INFO: normalize returned [0.5, 1.0, 1.5]


[0.5, 1.0, 1.5]

## 14. Input Validation Decorator

Catch bad inputs (wrong type, wrong shape) *before* they reach your model — a common cause of cryptic stack traces deep inside training loops.

In [16]:
import functools

def validate_types(**expected_types):
    """@validate_types(x=list, y=(int, float)) checks argument types by name."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            bound = dict(zip(func.__code__.co_varnames, args))
            bound.update(kwargs)
            for name, expected in expected_types.items():
                if name in bound and not isinstance(bound[name], expected):
                    raise TypeError(
                        f"Argument '{name}' must be {expected}, got {type(bound[name])}"
                    )
            return func(*args, **kwargs)
        return wrapper
    return decorator

@validate_types(features=list, learning_rate=(int, float))
def train_step(features, learning_rate=0.01):
    return f"Training with {len(features)} features, lr={learning_rate}"

print(train_step([0.1, 0.2, 0.3], learning_rate=0.001))

try:
    train_step("not_a_list", learning_rate=0.001)
except TypeError as e:
    print(f"Caught expected error: {e}")

Training with 3 features, lr=0.001
Caught expected error: Argument 'features' must be <class 'list'>, got <class 'str'>


## 15. Registry Pattern — Model-Zoo Style

This is exactly how libraries like **Hugging Face Transformers**, **timm**, **Detectron2**, and **PyTorch Lightning** let you register architectures/components by name, so you can instantiate them dynamically from a config file/string instead of hardcoding imports everywhere.

In [17]:
import functools

MODEL_REGISTRY = {}

def register_model(name):
    def decorator(func):
        MODEL_REGISTRY[name] = func
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)
        return wrapper
    return decorator

@register_model("resnet18")
def build_resnet18(num_classes=1000):
    return f"ResNet18(num_classes={num_classes})"

@register_model("simple_transformer")
def build_transformer(num_layers=6):
    return f"Transformer(num_layers={num_layers})"

print(MODEL_REGISTRY)

# Models can now be built purely from a config string, e.g. loaded from YAML/JSON
config = {"architecture": "resnet18", "num_classes": 10}
model = MODEL_REGISTRY[config["architecture"]](num_classes=config["num_classes"])
print(model)

{'resnet18': <function build_resnet18 at 0x7f173c26ec00>, 'simple_transformer': <function build_transformer at 0x7f173c26ed40>}
ResNet18(num_classes=10)


## 16. Singleton Decorator

Large ML models and tokenizers are expensive to load. A singleton decorator ensures a model/tokenizer is loaded **once** and reused everywhere — a common pattern in inference servers.

In [18]:
import functools

def singleton(cls):
    instances = {}
    @functools.wraps(cls)
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    return get_instance

@singleton
class ModelLoader:
    def __init__(self):
        print("Loading large model weights into memory... (expensive!)")
        self.weights = "huge-weights-blob"

loader1 = ModelLoader()
loader2 = ModelLoader()       # "Loading..." is NOT printed again
print(loader1 is loader2)     # True -- same instance reused

Loading large model weights into memory... (expensive!)
True


## 17. Thread-Safe Decorator

Inference servers (e.g. Flask/FastAPI serving a model) often handle requests across multiple threads. If a function mutates shared state, wrap it with a lock to avoid race conditions.

In [19]:
import functools
import threading

def synchronized(func):
    lock = threading.Lock()
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        with lock:
            return func(*args, **kwargs)
    return wrapper

request_count = 0

@synchronized
def increment_request_counter():
    global request_count
    request_count += 1
    return request_count

# Safe to call from multiple threads without race conditions
threads = [threading.Thread(target=increment_request_counter) for _ in range(10)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Final count: {request_count}")

Final count: 10


## 18. Type-Based Dispatch with `functools.singledispatch`

Sometimes you want a function to behave differently depending on the **type** of its first argument — e.g. a `preprocess()` function that handles raw text, a list of tokens, or a batch dict differently, without a chain of `isinstance` checks.

In [20]:
import functools

@functools.singledispatch
def preprocess(data):
    raise NotImplementedError(f"No preprocessing implemented for type {type(data)}")

@preprocess.register
def _(data: str):
    return data.lower().strip()

@preprocess.register
def _(data: list):
    return [preprocess(item) for item in data]

@preprocess.register
def _(data: dict):
    return {k: preprocess(v) for k, v in data.items()}

print(preprocess("  Hello WORLD  "))
print(preprocess(["  A ", " B  "]))
print(preprocess({"text": "  Mixed Case  "}))

hello world
['a', 'b']
{'text': 'mixed case'}


## 19. Async Decorators

Modern AI/ML services call LLM APIs, vector databases, or remote feature stores **asynchronously** for throughput. Decorating an `async def` function requires the wrapper itself to be `async` and to `await` the wrapped coroutine.

In [21]:
import asyncio
import functools
import time

def async_timer(func):
    @functools.wraps(func)
    async def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = await func(*args, **kwargs)
        print(f"[ASYNC TIMER] {func.__name__} took {time.perf_counter() - start:.3f}s")
        return result
    return wrapper

@async_timer
async def call_llm_async(prompt):
    await asyncio.sleep(0.3)          # simulate network latency
    return f"Response to: {prompt}"

async def main():
    results = await asyncio.gather(
        call_llm_async("prompt 1"),
        call_llm_async("prompt 2"),
        call_llm_async("prompt 3"),
    )
    print(results)

await main()   # Jupyter already runs an event loop, so top-level 'await' works here

[ASYNC TIMER] call_llm_async took 0.300s
[ASYNC TIMER] call_llm_async took 0.301s
[ASYNC TIMER] call_llm_async took 0.301s
['Response to: prompt 1', 'Response to: prompt 2', 'Response to: prompt 3']


## 20. Deprecation Warning Decorator

When an ML library evolves (e.g. renaming `model.predict_proba_old` to `model.predict_proba`), warn users instead of breaking their code outright.

In [22]:
import functools
import warnings

def deprecated(reason):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            warnings.warn(
                f"{func.__name__} is deprecated: {reason}",
                category=DeprecationWarning,
                stacklevel=2,
            )
            return func(*args, **kwargs)
        return wrapper
    return decorator

@deprecated("use `predict_proba` instead")
def predict_proba_old(x):
    return [0.1, 0.9]

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    predict_proba_old([1, 2, 3])
    print(caught[0].message)

predict_proba_old is deprecated: use `predict_proba` instead


## 21. Class Decorators

Decorators can also wrap an **entire class** rather than a single function — used for things like auto-registering model classes, injecting a generic `__repr__`, or adding validation hooks. (`dataclasses.dataclass` itself is a class decorator.)

In [23]:
def add_repr(cls):
    """Class decorator that injects a generic __repr__ based on instance attributes."""
    def __repr__(self):
        attrs = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"{cls.__name__}({attrs})"
    cls.__repr__ = __repr__
    return cls

@add_repr
class HyperParams:
    def __init__(self, lr, batch_size):
        self.lr = lr
        self.batch_size = batch_size

print(HyperParams(lr=0.001, batch_size=32))

HyperParams(lr=0.001, batch_size=32)


## 22. Typing Decorators Correctly with `ParamSpec` (PEP 612)

In production ML codebases that use `mypy`/`pyright`, a decorator written with bare `*args, **kwargs` erases type information for the decorated function — your IDE/type-checker can no longer tell you what arguments `predict()` actually expects. `ParamSpec` (Python 3.10+) fixes this and keeps full type-safety through the decorator.

In [24]:
from typing import Callable, TypeVar
try:
    from typing import ParamSpec            # Python 3.10+
except ImportError:
    from typing_extensions import ParamSpec  # pip install typing_extensions --break-system-packages
import functools
import time

P = ParamSpec("P")
R = TypeVar("R")

def typed_timer(func: Callable[P, R]) -> Callable[P, R]:
    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.perf_counter() - start:.4f}s")
        return result
    return wrapper

@typed_timer
def add(a: int, b: int) -> int:
    return a + b

# A type checker now correctly knows add(1, 2) -> int, and would flag
# add("a", "b") as a type error -- this information was lost with a
# plain *args/**kwargs wrapper (no ParamSpec).
print(add(2, 3))

add took 0.0000s
5


## 23. Mini Project — Production-Style Inference Pipeline

Let's combine several decorators into a realistic stack you might see wrapping a `predict()` function in a deployed ML service: **caching** + **logging** + **timing** + **retry**, all applied together.

Remember: decorators apply bottom-up, so **the order you stack them changes behavior**:
- `retry` should sit closest to the function, so failed attempts are retried *before* anything else sees them.
- `timer` around that measures the full duration including any retries.
- `log_calls` around that logs the call.
- `lru_cache` as the **outermost** layer, so a cache hit skips everything below it entirely (no logging, no timing, no retry logic re-executed).

In [25]:
import functools
import logging
import random
import time

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("inference_pipeline")

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        logger.info(f"{func.__name__} took {time.perf_counter() - start:.3f}s")
        return result
    return wrapper

def retry(times=3, delay=0.1, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as exc:
                    logger.warning(f"Attempt {attempt} failed: {exc!r}")
                    if attempt == times:
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.info(f"Calling {func.__name__}{args}")
        return func(*args, **kwargs)
    return wrapper

@functools.lru_cache(maxsize=64)          # outermost: skip everything below on a cache hit
@log_calls
@timer
@retry(times=3, delay=0.1, exceptions=(ConnectionError,))
def predict(prompt: str):
    if random.random() < 0.4:
        raise ConnectionError("simulated transient failure")
    time.sleep(0.2)                       # simulate model inference latency
    return f"prediction for '{prompt}'"

random.seed(1)   # seeded only so this demo prints reproducible output -- remove in real code
print(predict("cat photo"))
print(predict("cat photo"))               # second call: cache hit, instant, no logs from inside
print(predict.cache_info())

INFO: Calling predict('cat photo',)


INFO: predict took 0.301s


prediction for 'cat photo'
prediction for 'cat photo'
CacheInfo(hits=1, misses=1, maxsize=64, currsize=1)


## 24. Pitfalls & Best Practices

- **Always use `functools.wraps`** — otherwise you silently break introspection, documentation tools, and debugging.
- **Decoration happens once, at import/definition time.** Code in the "outer" layer of a parametrized decorator (outside `wrapper`) runs immediately on import, not on each call — be careful about placing expensive work there.
- **Order of stacked decorators matters** — always reason bottom-up, as in the mini project above.
- **Shared mutable state in decorators persists across all calls** unless explicitly scoped — be careful with module-level caches/counters. `lru_cache` on an instance method also holds a reference to `self`, which can leak memory if not handled carefully (e.g. by clearing the cache or using a bound per-instance cache instead).
- **Debug decorated functions** via `func.__wrapped__` to reach the original, un-decorated function.
- **Avoid reading volatile state (env vars, config) at decoration time** — read it inside `wrapper` instead, so it's re-evaluated on every call rather than baked in once at import.
- **Thread-safety:** decorators holding shared mutable state (counters, caches) need explicit locks when used inside multi-threaded inference servers.
- **Performance overhead:** each decorator layer adds a function call. For ultra-hot inner loops (millions of iterations) this overhead can matter — measure before decorating performance-critical paths.
- **Unhashable arguments break `lru_cache`** — never cache a function whose arguments include `numpy.ndarray` or `torch.Tensor` directly; convert to a hashable key first.

## 25. Cheat Sheet

| Decorator | Purpose | Typical AI/ML Use Case |
|---|---|---|
| Plain `def decorator(func): ...` | Add before/after behavior | Logging, validation |
| `decorator(arg)(func)` | Parametrized decorator | `@repeat(3)`, `@retry(times=5)` |
| Class-based (`__call__`) | Stateful decorator | Call counters, rate limiters |
| `functools.wraps` | Preserve metadata | Always — every decorator |
| `functools.lru_cache` / `cache` | Memoization | Caching embeddings, repeated inference |
| `@staticmethod` / `@classmethod` / `@property` | Method-level decorators | Model classes, `from_pretrained` constructors |
| `functools.singledispatch` | Type-based dispatch | `preprocess()` for str / list / dict |
| Registry decorator | Register into a dict | Model zoos, architecture registries |
| Singleton decorator | One shared instance | Loading large models/tokenizers once |
| `synchronized` (lock) decorator | Thread safety | Multi-threaded inference servers |
| `async def wrapper` | Async-aware decorator | Async LLM / vector-store API calls |
| `deprecated` decorator | Soft-deprecate APIs | Evolving ML library APIs |
| Class decorator | Modify/register a class | Auto `__repr__`, auto-registration |
| `ParamSpec`-typed decorator | Preserve type signatures | Type-safe production codebases |

### Further Reading
- PEP 318 — Decorators for Functions and Methods
- PEP 612 — Parameter Specification Variables (`ParamSpec`)
- The `functools` module documentation
- *Real Python: Primer on Python Decorators*

This covers decorators end-to-end — from the closures that make them possible, through to the exact registry, caching, retry, async, and typed-decorator patterns you'll find inside real AI/ML frameworks and production inference services.